# MuscleMap WB + MedSAM Refinement

Uses existing MuscleMap whole-body segmentations as bounding-box prompts for MedSAM.
For each fat-fraction image the corresponding `_dseg.nii.gz` is loaded; per-muscle
bounding boxes are extracted slice-by-slice and fed to MedSAM.

MedSAM embedding is computed **once per slice** and reused for all muscles.

**Kernel:** `dafne_clean`

In [1]:
import glob
import os
import numpy as np
import SimpleITK as sitk
import torch
from skimage import transform
from dafne.config import GlobalConfig
from dafne.utils.sam_mask_refine import (
    load_sam,
    medsam_inference,
    enlarge_bounding_box,
    determine_device,
)

C:\Users\docto\miniconda3\envs\dafne_clean\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
# --- paths ---
DEVICE      = determine_device()
MM_SEGS_DIR = "MuscleMap_segs"
IMAGE_GLOB  = "myosegmenTUM/*/ImageData/*FATFRACTION/*FATFRACTION_stack*.nii"
OUTPUT_DIR  = "MuscleMap_WB_medsam"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Device:", DEVICE)

SAM loaded on CPU
Device: cpu


In [3]:


# thigh	semimembranosus	left	7161
# thigh	semimembranosus	right	7162
# thigh	semitendinosus	left	7171
# thigh	semitendinosus	right	7172
# thigh	biceps femoris long head	left	7181
# thigh	biceps femoris long head	right	7182
# thigh	biceps femoris short head	left	7191
# thigh	biceps femoris short head	right	7192
# thigh	adductor magnus	left	7201
# thigh	adductor magnus	right	7202
# thigh	adductor longus	left	7211
# thigh	adductor longus	right	7212
# thigh	adductor brevis	left	7221
# thigh	adductor brevis	right	7222
# leg	anterior compartment	left	8101
# leg	anterior compartment	right	8102
# leg	deep posterior compartment	left	8111
# leg	deep posterior compartment	right	8112
# leg	lateral compartment	left	8121
# leg	lateral compartment	right	8122
# leg	soleus	left	8131
# leg	soleus	right	8132
# leg	gastrocnemius	left	8141
# leg	gastrocnemius	right	8142

In [4]:
# MuscleMap WB label map (from https://musclemap.github.io/MuscleMap/muscle-anatomy/)
# Convention: odd = left, even = right
LABEL_MAP = {
    7101: "Vastus_Lateralis_L",
    7102: "Vastus_Lateralis_R",
    7111: "Vastus_Intermedius_L",
    7112: "Vastus_Intermedius_R",
    7121: "Vastus_Medialis_L",
    7122: "Vastus_Medialis_R",
    7131: "Rectus_Femoris_L",
    7132: "Rectus_Femoris_R",
    7141: "Sartorius_L",
    7142: "Sartorius_R",
    7151: "Gracilis_L",
    7152: "Gracilis_R",
    7201: "Adductor_Magnus_L",
    7202: "Adductor_Magnus_R",
    7161: "Semimembranosus_L",
    7162: "Semimembranosus_R",
    7171: "Semitendinosus_L",
    7172: "Semitendinosus_R",
    7181: "Biceps_Femoris_L",
    7182: "Biceps_Femoris_R",
}
print(f"{len(LABEL_MAP)} muscle labels defined")

20 muscle labels defined


In [5]:
# load MedSAM once
GlobalConfig['SAM_MODEL'] = 'Med Sam'
sam_model = load_sam('Med Sam')
sam_model.eval()
print("MedSAM loaded on", DEVICE)

SAM loaded on CPU
MedSAM loaded on cpu


In [6]:
image_files = sorted(glob.glob(IMAGE_GLOB))
print(f"Found {len(image_files)} fat-fraction images")

# report which ones have a matching MuscleMap seg
matched, missing = [], []
for nii_path in image_files:
    stem = os.path.splitext(os.path.basename(nii_path))[0]
    seg_path = os.path.join(MM_SEGS_DIR, f"{stem}_dseg.nii.gz")
    if os.path.exists(seg_path):
        matched.append(nii_path)
    else:
        missing.append(stem)

print(f"  {len(matched)} have a MuscleMap seg, {len(missing)} do not")
if missing:
    print("  Missing segs for:", missing[:5], "..." if len(missing) > 5 else "")

Found 54 fat-fraction images
  54 have a MuscleMap seg, 0 do not


In [7]:
# run MuscleMap + MedSAM refinement slice-by-slice
for nii_path in matched:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f"{stem}_mm_medsam.npz")

    if os.path.exists(out_path):
        print(f"Skipping (already done): {out_path}")
        continue

    seg_path = os.path.join(MM_SEGS_DIR, f"{stem}_dseg.nii.gz")
    print(f"\nProcessing: {nii_path}")

    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(float)  # (slices, H, W)
    H, W = img_array.shape[1], img_array.shape[2]
    print(f"  Image shape: {img_array.shape}")

    seg_sitk  = sitk.ReadImage(seg_path)
    seg_array = sitk.GetArrayFromImage(seg_sitk)                # (slices, H, W), integer labels
    print(f"  Seg shape: {seg_array.shape}  labels present: {sorted(np.unique(seg_array[seg_array > 0]).tolist())}")

    all_masks = {}

    for slice_idx in range(img_array.shape[0]):
        slice_2d  = img_array[slice_idx]
        seg_slice = seg_array[slice_idx]

        # MedSAM embedding — computed once per slice
        img_norm   = slice_2d * 255.0 / (slice_2d.max() + 1e-8)
        img_3c     = np.repeat(img_norm[:, :, None], 3, axis=-1)
        img_1024   = transform.resize(
            img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True
        ).astype(np.uint8)
        img_1024   = (img_1024 - img_1024.min()) / np.clip(
            img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
        )
        img_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            image_embedding = sam_model.image_encoder(img_tensor)

        # refine each muscle using its MuscleMap bounding box
        for label_idx, muscle_name in LABEL_MAP.items():
            mask_arr = (seg_slice == label_idx).astype(np.uint8)

            if mask_arr.any():
                bbox     = enlarge_bounding_box(mask_arr)              # [min_col, min_row, max_col, max_row]
                box_1024 = bbox / np.array([W, H, W, H]) * 1024
                box_1024 = box_1024[None, None, :]                     # (1, 1, 4)
                refined  = medsam_inference(sam_model, image_embedding, box_1024, H, W)
            else:
                refined = mask_arr

            if muscle_name not in all_masks:
                all_masks[muscle_name] = np.zeros(img_array.shape, dtype=np.uint8)
            all_masks[muscle_name][slice_idx] = refined.astype(np.uint8)

        if (slice_idx + 1) % 5 == 0 or slice_idx == img_array.shape[0] - 1:
            print(f"  slice {slice_idx + 1}/{img_array.shape[0]} done")

    np.savez_compressed(out_path, **all_masks)
    print(f"  Saved → {out_path}")

print("\nAll done.")


Processing: myosegmenTUM\HV001_1\ImageData\HV001_1_FATFRACTION\HV001_1_FATFRACTION_stack1.nii
  Image shape: (65, 672, 672)
  Seg shape: (65, 672, 672)  labels present: [5121.0, 5122.0, 6101.0, 6102.0, 6111.0, 6112.0, 6121.0, 6122.0, 6131.0, 6132.0, 6141.0, 6142.0, 6151.0, 6152.0, 6160.0, 6171.0, 6172.0, 6181.0, 6182.0, 6191.0, 6192.0, 6201.0, 6202.0, 6211.0, 6212.0, 6221.0, 6222.0, 7101.0, 7102.0, 7111.0, 7112.0, 7121.0, 7122.0, 7131.0, 7132.0, 7141.0, 7142.0, 7151.0, 7152.0, 7161.0, 7171.0, 7172.0, 7181.0, 7182.0, 7201.0, 7202.0, 7211.0, 7212.0, 7221.0, 7222.0]
  slice 5/65 done
  slice 10/65 done
  slice 15/65 done
  slice 20/65 done
  slice 25/65 done
  slice 30/65 done
  slice 35/65 done
  slice 40/65 done
  slice 45/65 done
  slice 50/65 done
  slice 55/65 done
  slice 60/65 done
  slice 65/65 done
  Saved → MuscleMap_WB_medsam\HV001_1_FATFRACTION_stack1_mm_medsam.npz

Processing: myosegmenTUM\HV001_1\ImageData\HV001_1_FATFRACTION\HV001_1_FATFRACTION_stack2.nii
  Image shape: (6

KeyboardInterrupt: 

In [ ]:
# sanity check — reload one result
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.npz")))
if results:
    sample = np.load(results[0])
    print("Sample file:", results[0])
    for name in sample.files:
        arr = sample[name]
        print(f"  {name}: shape={arr.shape}  positive voxels={arr.sum()}")
else:
    print("No results yet.")